In [3]:
import os
import pandas as pd
import numpy as np
import pickle
import duckdb

In [ ]:
path_dataframe = 'data/2023_Yellow_Taxi_Trip_Data_20260526.csv'

# CONVERSÃO PARA PARQUET E CONVERSÃO DE TIPOS

In [ ]:
duckdb.sql("""
COPY (
    SELECT *
    FROM read_csv_auto(
        path_dataframe,
        all_varchar=true
    )
)
TO 'data/taxi_raw.parquet'
(FORMAT PARQUET)
""")

In [13]:
df = duckdb.sql("""
SELECT *
FROM read_parquet('data/taxi_raw.parquet')
LIMIT 5
""").df()

print(df.columns.tolist())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


In [14]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_clean AS
SELECT
    VendorID::INTEGER AS VendorID,

    STRPTIME(
        tpep_pickup_datetime,
        '%m/%d/%Y %I:%M:%S %p'
    ) AS tpep_pickup_datetime,

    STRPTIME(
        tpep_dropoff_datetime,
        '%m/%d/%Y %I:%M:%S %p'
    ) AS tpep_dropoff_datetime,

    passenger_count::INTEGER AS passenger_count,

    REPLACE(trip_distance, ',', '')::DOUBLE AS trip_distance,

    RatecodeID::INTEGER AS RatecodeID,

    store_and_fwd_flag,

    PULocationID::INTEGER AS PULocationID,
    DOLocationID::INTEGER AS DOLocationID,

    payment_type::INTEGER AS payment_type,

    REPLACE(fare_amount, ',', '')::DOUBLE AS fare_amount,
    REPLACE(extra, ',', '')::DOUBLE AS extra,
    REPLACE(mta_tax, ',', '')::DOUBLE AS mta_tax,
    REPLACE(tip_amount, ',', '')::DOUBLE AS tip_amount,
    REPLACE(tolls_amount, ',', '')::DOUBLE AS tolls_amount,
    REPLACE(improvement_surcharge, ',', '')::DOUBLE AS improvement_surcharge,
    REPLACE(total_amount, ',', '')::DOUBLE AS total_amount,
    REPLACE(congestion_surcharge, ',', '')::DOUBLE AS congestion_surcharge,
    REPLACE(airport_fee, ',', '')::DOUBLE AS airport_fee

FROM read_parquet('data/taxi_raw.parquet')
""")

In [15]:
duckdb.sql("""
COPY taxi_clean
TO 'data/taxi.parquet'
(FORMAT PARQUET)
""")

# TRATAMENTO DOS DADOS

Usecols = hour, tpep_pickup_time, PULocationID
Dropna

In [27]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand AS

SELECT
    DATE_TRUNC(
        'hour',
        tpep_pickup_datetime
    ) AS pickup_hour,
    
    PULocationID,
    COUNT(*) AS demand,

FROM 'data/taxi.parquet'

WHERE
    tpep_pickup_datetime IS NOT NULL
    AND PULocationID IS NOT NULL

GROUP BY
    pickup_hour,
    PULocationID

ORDER BY
    pickup_hour,
    PULocationID
""")

Criação de colunas descritivas a partir das colunas selecionadas previamente
novas_colunas = year, month, day, hour, weekday, is_weekend, covid_period

In [28]:
duckdb.sql("""
CREATE OR REPLACE TABLE taxi_demand_features AS

SELECT
    pickup_hour,
    PULocationID,
    demand,
    YEAR(pickup_hour) AS year,
    MONTH(pickup_hour) AS month,
    DAY(pickup_hour) AS day,
    HOUR(pickup_hour) AS hour,
    DAYOFWEEK(pickup_hour) AS weekday,
    CASE
        WHEN DAYOFWEEK(pickup_hour) IN (0, 6)
            THEN 1
        ELSE 0
    END AS is_weekend,

    CASE

        WHEN pickup_hour < TIMESTAMP '2020-03-15'
            THEN 'pre'

        WHEN pickup_hour < TIMESTAMP '2021-06-24'
            THEN 'during'

        ELSE 'post'

    END AS covid_period

FROM taxi_demand

ORDER BY
    pickup_hour,
    PULocationID
""")

Salva o parquet final de demanda

In [29]:
duckdb.sql("""
COPY taxi_demand_features
TO 'data/taxi_demand_processed.parquet'
(FORMAT PARQUET)
""")

In [30]:
print(
    duckdb.sql("""
    SELECT COUNT(*) AS total_rows
    FROM taxi_demand_features
    """).fetchall()
)

print(
    duckdb.sql("""
    SELECT *
    FROM taxi_demand_features
    LIMIT 10
    """).df()
)

[(886359,)]
          pickup_hour  PULocationID  demand  year  month  day  hour  weekday  \
0 2001-01-01 00:00:00           132       2  2001      1    1     0        1   
1 2001-01-01 00:00:00           161       1  2001      1    1     0        1   
2 2001-01-01 00:00:00           239       2  2001      1    1     0        1   
3 2001-01-01 15:00:00           246       1  2001      1    1    15        1   
4 2002-12-31 22:00:00           132       2  2002     12   31    22        2   
5 2002-12-31 23:00:00           132       6  2002     12   31    23        2   
6 2002-12-31 23:00:00           141       1  2002     12   31    23        2   
7 2002-12-31 23:00:00           237       1  2002     12   31    23        2   
8 2002-12-31 23:00:00           264       1  2002     12   31    23        2   
9 2003-01-01 00:00:00            79       1  2003      1    1     0        3   

   is_weekend covid_period  
0           0          pre  
1           0          pre  
2           0       